## Cohort retention

Retention was evaluated at D1, D7, D14, and D30 after signup. Key questions investigated:

- Which signup cohorts have the highest D30 retention?
- Does retention improve or decline across successive cohorts?
- How large is the drop-off from D1 to D7 and D30?
- Do newer cohorts exhibit materially different retention behavior?

In [ ]:
# Import Python packages
from snowflake.snowpark.context import get_active_session
import pandas as pd

# Get the current credentials
session = get_active_session()

In [ ]:
retention = session.table("PRODUCT_ANALYTICS.ANALYTICS.USER_RETENTION").to_pandas()

cohort_matrix = retention.pivot(
    index="SIGNUP_COHORT_MONTH",
    columns="DAYS_SINCE_SIGNUP",
    values="RETENTION_RATE"
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 7))

plt.imshow(cohort_matrix, aspect="auto")
plt.colorbar(label="Retention")
plt.xlabel("Days Since Signup")
plt.ylabel("Signup Cohort")
plt.title("User Retention by Signup Cohort")

plt.show()

The retention heatmap shows the expected triangular pattern because newer signup cohorts have not accumulated enough observation time to measure long-term retention.

To compare cohorts fairly, cohorts were therefore compared at fixed maturity points (particularly D7 and D30).

In [ ]:
retention_milestones = (
    retention[
        retention["DAYS_SINCE_SIGNUP"].isin([1, 7, 14, 30])
    ]
    .sort_values("SIGNUP_COHORT_MONTH")
    .pivot(
        index="SIGNUP_COHORT_MONTH",
        columns="DAYS_SINCE_SIGNUP",
        values="RETENTION_RATE"
    )
)

retention_milestones.columns = [
    f"D{col}" for col in retention_milestones.columns
]

retention_milestones

In [ ]:
retention[
    retention["DAYS_SINCE_SIGNUP"].isin([1, 7, 30])
].groupby("DAYS_SINCE_SIGNUP").apply(
    lambda x: (x["RETENTION_RATE"] * x["COHORT_USERS"]).sum()
              / x["COHORT_USERS"].sum()
)

### Which cohorts have the highest 30-day retention?

In [ ]:
d30 = (
    retention[
        retention["DAYS_SINCE_SIGNUP"] == 30
    ]
    .sort_values("RETENTION_RATE", ascending=False)
)

d30.head(10)

### Is retention improving over time?

Let's compare D7/D30 retention by cohort:

In [ ]:
retention[
    retention["DAYS_SINCE_SIGNUP"].isin([7, 30])
].pivot(
    index="SIGNUP_COHORT_MONTH",
    columns="DAYS_SINCE_SIGNUP",
    values="RETENTION_RATE"
).plot(
    marker="o",
    figsize=(10, 5)
)

### Do newer cohorts behave differently?

In [ ]:
d30 = retention[
    retention["DAYS_SINCE_SIGNUP"] == 30
].copy()

d30.groupby(
    "SIGNUP_COHORT_MONTH"
)["RETENTION_RATE"].mean().plot(
    marker="o",
    figsize=(10, 5)
)